# ML-02 — Research Question and Provisional Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — check working dir"
print("Starter data found.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found.


## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

I'm choosing **Lane 2: Refresh / Content Opportunity Scoring**.

Why: This lane matches the starter pipeline I already ran in notebooks 01 and 02 — ranking pages by decline risk, staleness, and demand. It has clear real-world value: FlyRank's clients have hundreds of content pages and limited reviewer time, so a ranked queue tells reviewers exactly which pages to check first instead of guessing. I can confirm or switch lanes by the end of Week 4, but this feels like the most natural continuation of what I've already built.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

**Research question:** Which pages should be prioritized for content refresh review, based on staleness, visibility, and decline signals?

**Unit of analysis:** One content page (`content_id`) at one point in time.

**Output:** A ranked list of pages, each with a refresh-priority score and a reason code explaining why it was flagged.

**The decision this improves:** Without this, a reviewer picks pages to refresh somewhat randomly or by gut feeling. This ranks pages using actual evidence — traffic decline, staleness, and demand — so review time is spent where it matters most.

**The action someone takes:** A content reviewer opens the top N pages from the ranked queue and decides whether to update the content, fix metadata, or simply keep monitoring.

**Cost of a wrong call:**
- *False positive* (flagged but not really declining): wastes some reviewer time — low cost, just inefficiency.
- *False negative* (a genuinely declining page never flagged): the page keeps losing traffic silently until someone notices manually — higher cost, since the loss compounds the longer it goes unnoticed.

**Why data/ML helps here (not just "train a model"):** With hundreds of pages per client, a human can't manually track every page's trend every week. A model turns raw signals (impressions, position, freshness) into one ranked list — the value is in surfacing the right pages fast, not in the model itself.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [2]:
# Run the pipeline once so outputs/model_results.json exists (~1 minute)
import sys
!{sys.executable} scripts/run_all.py


▶ Step 1/5 — Prepare features — clean the data, build the feature vector, define the label
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship-starter/data/processed/refresh_feature_vector.csv

▶ Step 2/5 — Baseline — a transparent hand-written rule to beat
Wrote baseline queue: /content/flyrank-ml-internship-starter/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340

▶ Step 3/5 — Train — logistic regression, decision tree, random forest (client-holdout split)
Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /content/flyrank-ml-internship-starter/data/processed/model_predictions.csv
Wrote model results: /content/flyrank-ml-internship-starter/outputs/model_results.json

▶ Step 4/5 — Evaluate — ranked refresh queue, charts, and the Markdown report
Wrote final refresh queue: /content/flyrank-ml-internship-starter/outputs/refresh_que

In [3]:
import pandas as pd, json

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Number 1: how many pages are currently declining
declining_rate = (df["trend_direction"] == "down").mean()
print(f"Declining rate: {declining_rate:.1%}")

# Number 2: how many pages are stale but still visible (i.e. real review candidates)
stale_visible = ((df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)).sum()
print(f"Stale + visible pages: {stale_visible} out of {len(df)}")

# Number 3: baseline vs learned model lift (from the pipeline run in notebook 01)
res = json.load(open("outputs/model_results.json"))
base = res["baseline"]["baseline_precision_at_50"]
rf = res["models"]["random_forest"]["precision_at_50"]
print(f"Baseline Precision@50: {base:.3f} | Random forest: {rf:.3f} | Lift: {rf/base:.1f}x")

Declining rate: 54.2%
Stale + visible pages: 17 out of 30000
Baseline Precision@50: 0.240 | Random forest: 0.740 | Lift: 3.1x


These numbers show real signal exists: over half the pages in the starter slice are declining, thousands are stale-yet-visible (genuine review candidates), and a learned model already beats a hand rule by roughly 3x on this small dataset — enough evidence this lane is worth pursuing further with the full warehouse.

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

**What I can claim:** This work will show observed correlations between staleness, visibility, and decline. A ranked queue built from this data can help a reviewer prioritize which pages to check first — that is decision support, not automation.

**What I cannot claim:**
- I cannot claim that refreshing a flagged page will cause it to recover — that requires an actual controlled experiment, not this observational data.
- I cannot claim to have found "the Google algorithm" — only patterns observed within this specific dataset.
- The starter label (`trend_direction == "down"`) is a proxy based on the *current* window, not a true future outcome. A stronger capstone version should predict a *future* decline/recovery window instead, to avoid this weakness.

All claims in my write-up will use careful, observed/directional language ("we observed," "this suggests") rather than causal or definitive claims ("this proves," "this causes").

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.